# Polynomial Fitting: A Reproducible Pipeline

A minimal example of a parameterised data science pipeline.  
Change the values in the **Parameters** cell below, re-run top-to-bottom, and observe how results change.

This is the pipeline we will version with DVC in the workshop.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Parameters (change these and re-run) ---
RANDOM_SEED   = 42
NUM_POINTS    = 100
NOISE_LEVEL   = 0.5
X_MIN         = -3.0
X_MAX         =  3.0
TRUE_COEFFS   = [0.5, -2.0, 1.0]  # descending degree: 0.5x² - 2x + 1
POLY_DEGREE   = 2

: 

In [ ]:
def generate_data():
    rng = np.random.default_rng(RANDOM_SEED)
    x = np.linspace(X_MIN, X_MAX, NUM_POINTS)
    y_true = np.polyval(TRUE_COEFFS, x)
    y_noisy = y_true + rng.normal(0, NOISE_LEVEL, NUM_POINTS)
    return x, y_true, y_noisy


def fit_polynomial(x, y, degree):
    coeffs = np.polyfit(x, y, degree)
    y_pred = np.polyval(coeffs, x)
    return coeffs, y_pred


def evaluate(y, y_pred):
    mse = np.mean((y - y_pred) ** 2)
    rmse = np.sqrt(mse)
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot
    return {'MSE': mse, 'RMSE': rmse, 'R2': r2}

## 1. Generate Data

Synthetic data from a known polynomial plus Gaussian noise.  
The true function is shown as a dashed line — the goal of fitting is to recover it.

In [ ]:
x, y_true, y_noisy = generate_data()

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x, y_noisy, alpha=0.5, s=30, color='#3498db', label='Noisy data')
ax.plot(x, y_true, 'k--', linewidth=2, label='True function')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Generated Data')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'True degree:       {len(TRUE_COEFFS) - 1}')
print(f'True coefficients: {TRUE_COEFFS}')
print(f'Noise level:       {NOISE_LEVEL}')
print(f'Points:            {NUM_POINTS}')

## 2. Fit Polynomial

Least-squares polynomial fit at the degree set by `POLY_DEGREE`.  
When `POLY_DEGREE` matches the true degree, the recovered coefficients should be close to `TRUE_COEFFS`.

In [ ]:
coeffs, y_pred = fit_polynomial(x, y_noisy, POLY_DEGREE)

x_smooth = np.linspace(X_MIN, X_MAX, 300)
y_smooth = np.polyval(coeffs, x_smooth)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x, y_noisy, alpha=0.5, s=30, color='#3498db', label='Noisy data')
ax.plot(x, y_true, 'k--', linewidth=2, label='True function')
ax.plot(x_smooth, y_smooth, color='#e74c3c', linewidth=2.5, label=f'Fit (degree {POLY_DEGREE})')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f'Polynomial Fit — Degree {POLY_DEGREE}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Fitted (degree {POLY_DEGREE}): {[round(c, 4) for c in coeffs]}')
if POLY_DEGREE == len(TRUE_COEFFS) - 1:
    print(f'True   (degree {POLY_DEGREE}): {TRUE_COEFFS}')

## 3. Evaluate

Metrics on the noisy data and a residual plot to check for systematic errors.

In [ ]:
metrics = evaluate(y_noisy, y_pred)

print('Metrics')
print('-' * 25)
for name, val in metrics.items():
    print(f'  {name:<6} {val:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.scatter(y_pred, y_noisy, alpha=0.5, s=20, color='#3498db')
lims = [min(y_pred.min(), y_noisy.min()), max(y_pred.max(), y_noisy.max())]
ax1.plot(lims, lims, 'r--', linewidth=1.5)
ax1.set_xlabel('Predicted')
ax1.set_ylabel('Actual')
ax1.set_title('Actual vs Predicted')
ax1.grid(True, alpha=0.3)

residuals = y_noisy - y_pred
ax2.scatter(x, residuals, alpha=0.5, s=20, color='#e67e22')
ax2.axhline(0, color='r', linestyle='--', linewidth=1.5)
ax2.set_xlabel('x')
ax2.set_ylabel('Residual')
ax2.set_title('Residuals')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Compare Degrees

The same data fitted at several degrees. Low degrees underfit; high degrees overfit.

The current `POLY_DEGREE` is highlighted in yellow.

In [ ]:
degrees = sorted({1, 2, 3, 5, POLY_DEGREE})

n_deg = len(degrees)
n_cols = min(n_deg, 3)
n_rows = (n_deg + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), squeeze=False)
axes = axes.flatten()

x_smooth = np.linspace(X_MIN, X_MAX, 300)

for i, deg in enumerate(degrees):
    ax = axes[i]
    c, yp = fit_polynomial(x, y_noisy, deg)
    m = evaluate(y_noisy, yp)
    ax.scatter(x, y_noisy, alpha=0.35, s=15, color='#3498db')
    ax.plot(x, y_true, 'k--', linewidth=1.5, alpha=0.5)
    ax.plot(x_smooth, np.polyval(c, x_smooth), color='#e74c3c', linewidth=2)
    ax.set_title(f'Degree {deg}  |  R2 = {m["R2"]:.3f}')
    ax.set_xlabel('x')
    ax.grid(True, alpha=0.3)
    if deg == POLY_DEGREE:
        ax.set_facecolor('#fef9e7')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Polynomial Fits at Different Degrees', fontsize=13)
plt.tight_layout()
plt.show()